# Ares Capital (ARCC): an edgartools walkthrough VERY INCOMPLETE

This notebook retrieves Ares Capital's most recently filed 10-Q directly from SEC EDGAR, then turns filing and XBRL data into pandas DataFrames. Run the cells in order.

In [7]:
import os
from pathlib import Path

# Keep edgartools download/cache files inside this project.
os.environ['EDGAR_LOCAL_DATA_DIR'] = str(Path.cwd() / '.edgar-data')

import pandas as pd
import edgar

edgar.set_identity('tah428@lehigh.edu')

print(f'edgartools version: {edgar.__version__}')

edgartools version: 5.55.0


## 1. Find a company

`Company` accepts a ticker or CIK. Ares Capital's ticker is `ARCC`.

In [8]:
arcc = edgar.Company('ARCC')
company_summary = pd.DataFrame([{
    'name': arcc.name,
    'cik': arcc.cik,
    'tickers': ', '.join(arcc.tickers),
    'sic': arcc.sic,
}])
company_summary

,name,cik,tickers,sic
0,ARES CAPITAL CORP,1287750,ARCC,


## 2. Get the most recent 10-Q

`get_filings` retrieves the filing index; `latest()` selects the newest row. This means the notebook stays current each time you run it.

In [9]:
ten_q_filings = arcc.get_filings(form='10-Q', amendments=False)
latest_10q_filing = ten_q_filings.latest()

# Filing-index data is easy to bring into pandas for a research table.
ten_q_history = ten_q_filings.to_pandas()
ten_q_history.head(5)

,accession_number,filing_date,reportDate,acceptanceDateTime,act,form,fileNumber,items,size,isXBRL,isInlineXBRL,primaryDocument,primaryDocDescription
0,0001628280-26-050307,2026-07-29,2026-06-30,2026-07-29 00:51:40+00:00,34,10-Q,814-00663,,76759086,1,1,arcc-20260630.htm,10-Q
1,0001628280-26-027688,2026-04-28,2026-03-31,2026-04-28 01:09:50+00:00,34,10-Q,814-00663,,68076731,1,1,arcc-20260331.htm,10-Q
2,0001287750-25-000046,2025-10-28,2025-09-30,2025-10-28 00:48:29+00:00,34,10-Q,814-00663,,64393483,1,1,arcc-20250930.htm,10-Q
3,0001287750-25-000038,2025-07-29,2025-06-30,2025-07-29 01:23:20+00:00,34,10-Q,814-00663,,55303123,1,1,arcc-20250630.htm,10-Q
4,0001287750-25-000020,2025-04-29,2025-03-31,2025-04-29 01:50:25+00:00,34,10-Q,814-00663,,53955028,1,1,arcc-20250331.htm,10-Q


In [ ]:
latest_10q_filing

# Use .obj() to turn the filing into edgartools' structured TenQ object.
latest_10q = latest_10q_filing.obj() ## Standard way to make it easier to go thorugh the filing's data.
print('Form:', latest_10q.form)
print('Filed:', latest_10q.filing_date)
print('Period ended:', latest_10q.period_of_report)

Form: 10-Q
Filed: 2026-07-29
Period ended: 2026-06-30


## 3. Inspect financial statements

The TenQ object exposes the filing's XBRL financials. These rendered tables are a quick way to explore the statement structure before doing analysis.

In [11]:
financials = latest_10q.financials

# Try one at a time: income_statement(), balance_sheet(), cash_flow_statement().
financials.balance_sheet()

                                                                                                                   
                                         ARES CAPITAL CORPORATION   ARCC                                           
                                         CONSOLIDATED BALANCE SHEETS                                               
                                         Dec 31, 2025 to Jun 30, 2026                                              
                                                                                                                   
                                                                                    Jun 30, 2026   Dec 31, 2025    
   ─────────────────────────────────────────────────────────────────────────────────────────────────────────────   
        ASSETS                                                                                                     
          Investments at fair value                                     

## 4. Collect data into pandas

For analysis, company-level statements provide a clean multi-quarter table, while `get_facts()` gives the raw XBRL facts for custom research.

In [16]:
# Four reported quarterly periods as a DataFrame.
quarterly_balance_sheet = arcc.balance_sheet(
    period='quarterly', periods=4, as_dataframe=True
)
quarterly_balance_sheet

,label,depth,is_abstract,is_total,section,confidence,Q1 2026,Q3 2025,Q2 2025,Q1 2025
concept,,,,,,,,,,
AssetsAbstract,ASSETS,0,True,False,NaN,0.836858,NaN,NaN,NaN,NaN
Assets,Assets,1,False,True,NaN,0.900302,3.067900e+10,3.080600e+10,2.907100e+10,2.831700e+10
AssetsCurrentAbstract,Current assets:,1,True,False,NaN,0.516616,NaN,NaN,NaN,NaN
CashAndCashEquivalentsAtCarryingValue,"Cash and Cash Equivalents, at Carrying Value",2,False,False,NaN,0.728097,5.050000e+08,1.036000e+09,4.470000e+08,6.470000e+08
LiabilitiesAndStockholdersEquityAbstract,LIABILITIES AND EQUITY,0,True,False,NaN,0.716012,NaN,NaN,NaN,NaN
Liabilities,Liabilities,1,False,True,NaN,0.728097,1.661400e+10,1.648400e+10,1.503700e+10,1.464500e+10
StockholdersEquityAbstract,Stockholders’ equity:,1,True,False,NaN,0.592145,NaN,NaN,NaN,NaN
CommonStockValue,"Common Stock, Value, Issued",2,False,False,NaN,0.770393,1.000000e+06,1.000000e+06,1.000000e+06,1.000000e+06
RetainedEarningsAccumulatedDeficit,Retained Earnings (Accumulated Deficit),2,False,False,NaN,0.836858,7.050000e+08,8.510000e+08,7.890000e+08,7.650000e+08


In [31]:
# Raw XBRL facts are useful when the standard statement doesn't include a BDC-specific metric.
facts = arcc.get_facts().to_dataframe(include_metadata=True)
facts[['concept', 'label', 'numeric_value', 'unit', 'period_end', 'form_type']].head(10)

,concept,label,numeric_value,unit,period_end,form_type
0,cef:AcquiredFundFeesAndExpensesPercent,,0.0279,number,2024-04-24,424B2
1,cef:AcquiredFundFeesAndExpensesPercent,,0.0146,number,2026-04-28,424B2
2,cef:AcquiredFundFeesAndExpensesPercent,,0.0279,number,2026-05-04,424B2
3,cef:DividendReinvestmentAndCashPurchaseFees,,15.0000,USD,2026-04-28,424B2
4,cef:DividendReinvestmentAndCashPurchaseFees,,15.0000,USD,2026-05-04,424B2
5,cef:ExpenseExampleYear01,,-123.0000,USD,2026-04-28,424B2
6,cef:ExpenseExampleYear01,,-123.0000,USD,2026-05-04,424B2
7,cef:ExpenseExampleYears1to10,,-894.0000,USD,2026-04-28,424B2
8,cef:ExpenseExampleYears1to10,,-894.0000,USD,2026-05-04,424B2
9,cef:ExpenseExampleYears1to3,,-342.0000,USD,2026-04-28,424B2


In [32]:
# Example: search raw facts for investment-related concepts, then save the result for later analysis.
investment_facts = (
    facts.loc[facts['label'].str.contains('investment', case=False, na=False)]
    .sort_values(['period_end', 'concept'], ascending=[False, True])
    [['concept', 'label', 'numeric_value', 'unit', 'period_end', 'form_type']]
)
investment_facts.head(25)

,concept,label,numeric_value,unit,period_end,form_type
229,us-gaap:AccretionAmortizationOfDiscountsAndPre...,Accretion (Amortization) of Discounts and Prem...,4.000000e+06,USD,2026-03-31,10-Q
1853,us-gaap:GrossInvestmentIncomeOperating,"Gross Investment Income, Operating",7.630000e+08,USD,2026-03-31,10-Q
2387,us-gaap:InvestmentCompanyDistributionToShareho...,"Investment Company, Distribution to Shareholde...",4.800000e-01,USD per share,2026-03-31,10-Q
2446,us-gaap:InvestmentCompanyDividendDistribution,"Investment Company, Dividend Distribution",3.450000e+08,USD,2026-03-31,10-Q
2508,us-gaap:InvestmentCompanyExpenseRatioIncluding...,"Investment Company, Expense Ratio Including In...",1.027000e-01,number,2026-03-31,10-Q
2536,us-gaap:InvestmentCompanyFinancialCommitmentTo...,"Investment Company, Financial Commitment to In...",3.300000e+07,USD,2026-03-31,10-Q
2598,us-gaap:InvestmentCompanyGainLossOnInvestmentP...,"Investment Company, Gain (Loss) on Investment,...",-4.200000e-01,USD per share,2026-03-31,10-Q
2660,us-gaap:InvestmentCompanyInvestmentIncomeLossF...,"Investment Company, Investment Income (Loss) f...",1.300000e-01,USD per share,2026-03-31,10-Q
2722,us-gaap:InvestmentCompanyInvestmentIncomeLossP...,"Investment Company, Investment Income (Loss), ...",5.500000e-01,USD per share,2026-03-31,10-Q
2784,us-gaap:InvestmentCompanyInvestmentIncomeLossR...,"Investment Company, Investment Income (Loss) R...",1.136000e-01,number,2026-03-31,10-Q


In [33]:
finder = latest_10q_filing.grep("Schedule of Investments")
finder

╭─────────────────────── grep 'Schedule of Investments' (4514 matches) — showing first 20 ────────────────────────╮
│ ┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓ │
│ ┃ Location        ┃ Context                                                                                   ┃ │
│ ┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩ │
│ │ EX-99.1         │ ...CONSOLIDATED CONDENSED SCHEDULE OF INVESTMENTS...                                      │ │
│ │ EX-99.1         │ ...CONSOLIDATED CONDENSED SCHEDULE OF INVESTMENTS...                                      │ │
│ │ EX-101.SCH      │ ...TEDSCHEDULEOFINVESTMENTSParenthetical">                                                │ │
│ │                 │         <link:definition>0000005 - Statement - CONSOLIDATED SCHEDULE OF INVESTMENTS -     │ │
│ │                 │ (Parenthetical)</link:definition>                 

In [ ]:
finder[0].

EX-99.1:  ...CONSOLIDATED CONDENSED SCHEDULE OF INVESTMENTS...